# 01- Experiment Objective

# 02 Experimental Definition

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [ ]:
from src.core.utility_experiment_config import UtilityExperimentConfig
from src.core.dataset_config import DatasetConfig
from src.core.preprocessing_config import PreprocessingConfig
from src.core.task_config import TaskConfig


In [ ]:
CATEGORICAL_COLUMNS_FOR_CLASSIFICATION = [
    "TP_COR_RACA",
    "TP_NACIONALIDADE",
    "TP_ESTADO_CIVIL",
    "TP_ST_CONCLUSAO",
    "TP_ENSINO",
    "IN_TREINEIRO",
    "Q001",
    "Q002",
    "Q003",
    "Q004",
    "Q005",
    "Q006",
    "Q007",
    "Q008",
    "Q009",
    "Q010",
    "Q011",
    "Q012",
    "Q013",
    "Q014",
    "Q015",
    "Q016",
    "Q017",
    "Q018",
    "Q019",
    "Q020",
    "Q021",
    "Q022",
    "Q023",
]

NUMERICAL_COLUMNS_FOR_CLASSIFICATION = [
    "TP_FAIXA_ETARIA",
    "TP_ANO_CONCLUIU",
]

In [ ]:
CATEGORICAL_COLUMNS_FOR_REGRESSION = [
    "TP_SEXO",
    "TP_COR_RACA",
    "TP_NACIONALIDADE",
    "TP_ESTADO_CIVIL",
    "TP_ST_CONCLUSAO",
    "TP_ENSINO",
    "IN_TREINEIRO",
    "Q001",
    "Q002",
    "Q003",
    "Q004",
    "Q006",
    "Q007",
    "Q008",
    "Q009",
    "Q010",
    "Q011",
    "Q012",
    "Q013",
    "Q014",
    "Q015",
    "Q016",
    "Q017",
    "Q018",
    "Q019",
    "Q020",
    "Q021",
    "Q022",
    "Q023",
]

NUMERICAL_COLUMNS_FOR_REGRESSION = [
    "TP_FAIXA_ETARIA",
    "TP_ANO_CONCLUIU",
]

# Experiment Config

In [ ]:
def get_regression_config():

    dataset_config = DatasetConfig(
        dataset_name="enem",
        dataset_version="enem_2025 - v-2026-09-08_00-56-16",
        data_sample_size=100_000,
        data_random_state=42,
    )
        

    preprocessing_config = PreprocessingConfig(
        categorical_columns=CATEGORICAL_COLUMNS_FOR_REGRESSION,
        numerical_columns=NUMERICAL_COLUMNS_FOR_REGRESSION,
    )

    return UtilityExperimentConfig(
        dataset=dataset_config,
        task= TaskConfig(task_type="regression", target="Q005"),
        preprocessing=preprocessing_config,
    )

In [ ]:
def get_classification_config():

    dataset_config = DatasetConfig(
        dataset_name="enem",
        dataset_version="enem_2025 - v-2026-09-08_00-56-16",
        data_sample_size=100_000,
        data_random_state=42,
    )


    preprocessing_config = PreprocessingConfig(
        categorical_columns=CATEGORICAL_COLUMNS_FOR_CLASSIFICATION,
        numerical_columns=NUMERICAL_COLUMNS_FOR_CLASSIFICATION,
    )

    return UtilityExperimentConfig(
        dataset=dataset_config,
        task=TaskConfig(task_type="classification", target="TP_SEXO"),
        preprocessing=preprocessing_config,
    )

# Select experiment

In [ ]:


evaluation_config = get_regression_config()

#evaluation_config = get_classification_config()


03 Dataset Preparation

In [ ]:
from src.data.dataset_registry import load_dataset_bundle


In [ ]:
dataset_bundle = load_dataset_bundle(
    dataset_name=evaluation_config.dataset.dataset_name,
    dataset_version=evaluation_config.dataset.dataset_version,
    columns=evaluation_config.preprocessing.categorical_columns 
    + evaluation_config.preprocessing.numerical_columns
    + [evaluation_config.task.target],
    sample_size=evaluation_config.dataset.data_sample_size,
    random_state=evaluation_config.dataset.data_random_state,
)

dataset_bundle

04 Feature Preparation

In [ ]:
from src.core.splits_config import SplitConfig

split_plan = SplitConfig(seed=42, test_size=0.5)

In [ ]:
from src.experiments.utility_evaluation_services import feature_preparation

prepared_features = []

for dataset_name, df in zip(dataset_bundle['dataset_names'], dataset_bundle['datasets']):
    
    prepared = feature_preparation.prepare_features(
        name=dataset_name,
        df=df,
        task_config=evaluation_config.task,
        split_plan=split_plan,
        preprocessing_config=evaluation_config.preprocessing,
    )
    prepared_features.append(prepared)


print(f"Prerpared features: {len(prepared_features)}")

05 Model execution

In [ ]:
from src.core.models_spec_config import ModelSpec

# Classification Models

In [ ]:
XGBOOST_CLASSIFIER = ModelSpec(
    name="xgboost",
    model_type="xgboost_classifier",
    parameters={
        "n_estimators": 500,
        "max_depth": 4,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 1,
        "gamma": 0.0,
        "reg_alpha": 0.0,
        "reg_lambda": 5.0,
        "tree_method": "hist",
        "n_jobs": 4,
        "verbosity": 0,
    },
)


# Regression Models

In [ ]:
XGBOOST_REGRESSOR = ModelSpec(
    name="xgboost",
    model_type="xgboost_regressor",
    parameters={
        "objective": "reg:squarederror",
        "n_estimators": 500,
        "max_depth": 7,
        "learning_rate": 0.05,
        "subsample": 0.8,
        "colsample_bytree": 0.8,
        "min_child_weight": 1,
        "gamma": 0.0,
        "reg_alpha": 0.0,
        "reg_lambda": 1.0,
        "tree_method": "hist",
        "n_jobs": 4,
        "verbosity": 0,
    },
)

In [ ]:
CLASSIFICATION_MODELS = [
    XGBOOST_CLASSIFIER,
]


REGRESSION_MODELS = [
    XGBOOST_REGRESSOR,
]

# Running models

In [ ]:
from src.experiments.utility_evaluation_services.model import model_runner

06 Utility Evaluation

In [ ]:
from src.experiments.utility_evaluation_services import metrics

from src.core.results_config import UtilityClassificationResult, UtilityRegressionResult

from dataclasses import asdict

utility_records = []
leakage_input= []

for prepared in prepared_features:

    models = (
        CLASSIFICATION_MODELS
        if prepared.task_type == "classification"
        else REGRESSION_MODELS
    )

    for model_spec in models:

        prediction = model_runner.execute_model(
            prepared_features=prepared,
            model_spec=model_spec,
        )

        utility = metrics.compute_utility_metrics(
            prediction_result=prediction,
            task_type=prepared.task_type,
        )

        record  = {
            "dataset": prepared.name,
            "task_type": prepared.task_type,
            "target": prepared.target,
            "model": model_spec.name,
            "model_type": model_spec.model_type,
        }

        record.update(asdict(utility))

        if type(utility)  == UtilityClassificationResult:

            leakage_input.append({
                "dataset": prepared.name,
                "task_type": prepared.task_type,
                "target": prepared.target,
                "model": model_spec.name,
                "model_type": model_spec.model_type,

                "X_pool": prepared.X_train,
                "y_pool": prepared.y_train,

                "target_prediction": {
                    "train_proba": prediction.train_proba,
                    "test_proba": prediction.test_proba,
                    "y_train_encoded": prediction.y_train_encoded,
                    "y_test_encoded": prediction.y_test_encoded,
                },
    })

            
        elif type(utility) == UtilityRegressionResult:
            leakage_input.append({
                "dataset": prepared.name,
                "task_type": prepared.task_type,
                "target": prepared.target,
                "model": model_spec.name,
                "model_type": model_spec.model_type,

                "X_pool": prepared.X_train,
                "y_pool": prepared.y_train,

                "target_prediction": {
                    "y_train_true": prediction.y_train_true,
                    "y_train_pred": prediction.y_train_pred,
                    "y_test_true": prediction.y_test_true,
                    "y_test_pred": prediction.y_test_pred,
                },
            })

        utility_records.append(record)


utility_records

07 Visualization

08 Export

In [ ]:
import pandas as pd
from datetime import datetime

from artifacts.persistence import persist_utility_artifact


ARTIFACT_SCHEMA_VERSION = "1.0"

experiment_id =  datetime.now().strftime("%Y%m%d_%H%M%S")

utility_metrics = pd.DataFrame(utility_records)
utility_metrics.insert(0, "experiment_id", experiment_id)

input_leakage= pd.DataFrame(leakage_input)

if evaluation_config.task.task_type == "classification":

    EXPERIMENT_TYPE = "classification_evaluation"

    model_specs = {
        (model.name, model.model_type): {
            "name": model.name,
            "model_type": model.model_type,
            "parameters": model.parameters,
        }
        for model in [*CLASSIFICATION_MODELS]
    }

elif evaluation_config.task.task_type == "regression":

    EXPERIMENT_TYPE = "regression_evaluation"

    model_specs = {
        (model.name, model.model_type): {
            "name": model.name,
            "model_type": model.model_type,
            "parameters": model.parameters,
        }
        for model in [*REGRESSION_MODELS]
    }

experiment_metadata = {
    "experiment_id": experiment_id,
    "experiment_type": EXPERIMENT_TYPE,
    "artifact_schema_version": ARTIFACT_SCHEMA_VERSION,
    "created_at": datetime.now().isoformat(timespec="seconds"),
    "dataset": {
        "name": evaluation_config.dataset.dataset_name,
        "version": evaluation_config.dataset.dataset_version,
        "sample_size": evaluation_config.dataset.data_sample_size,
        "random_state": evaluation_config.dataset.data_random_state,
    },
    "split": {
        "seed": split_plan.seed,
        "test_size": split_plan.test_size,
    },
    "preprocessing": {
        "categorical_columns": evaluation_config.preprocessing.categorical_columns,
        "numerical_columns": evaluation_config.preprocessing.numerical_columns,
    },
    "tasks": {"task_type": evaluation_config.task.task_type, "target": evaluation_config.task.target},
    "models": list(model_specs.values()),
}

artifact_path = persist_utility_artifact(
    experiment_id=experiment_id,
    metadata=experiment_metadata,
    utility_metrics=utility_metrics,
    input_leakage= input_leakage
)

artifact_path

10 Conclusion / Observations